# Fine-tuning a Language Model for Your Texts

This tutorial walks through the complete workflow for training a spaCy language model on your own annotated text. By the end you will have a model trained to tag, lemmatize, and parse your texts more accurately than spaCy's generic English model.

We will use Shakespeare's *The Winter's Tale* as the example corpus throughout. Substitute your own CONLL-U file wherever you see `wt_sanitized.conllu`.

## What you need before starting

- A CONLL-U formatted treebank file (the training data). If your data is in a different format, you will need to convert it first — spaCy's documentation covers this.
- A Python environment with spaCy, cupy (optional, for GPU), and this module installed. See `README.md` for the setup commands.
- Patience: training takes time. On a modern GPU, a small corpus (~300 sentences) trains in a few minutes. On CPU it takes 20–40 minutes.

In [1]:
# Add the parent directory to the path so Python can find the language_model module.
# This is needed because the notebook lives inside language_model/ itself.
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

In [2]:
from language_model import LanguageModel, split_conllu

c:\Users\aaron\OneDrive\Desktop\Lexos-LM-Workflow\Lexos_LM_Fine-tuning_Workflow\.venv\Lib\site-packages\cupy\_environment.py:284: UserWarning: CUDA path could not be detected. Set CUDA_PATH environment variable if CuPy fails to load.
  warnings.warn(


---

## Configuration

This cell defines the source models used as starting points for fine-tuning. The module ships with a pre-trained English UD model (`pretrained/ud_en_ewt`) for the three components that spaCy's built-in models do not cover. Change any path here to use a different source model.

In [3]:
# --- Configuration ---
# Path to the UD-trained English model bundled with this module.
# Used as the default starting point for morphologizer, trainable_lemmatizer,
# and parser (components that en_core_web_sm does not cover in UD form).
# Change this if you want to use a different source model for those components.
UD_BASE_MODEL = str(Path().resolve() / "pretrained" / "ud_en_ewt" / "model-best")

---

## Step 1: Split your data

Training a model requires three separate datasets:

- **Train** — the data the model actually learns from (largest portion)
- **Dev** — used during training to check progress and decide when to stop (prevents overfitting)
- **Test** — held back entirely until the end, used for your final accuracy measurement

If you already have three separate files you can skip this step and pass them directly to `copy_assets()`. If you have a single file, `split_conllu` handles the split.

The default 80/10/10 split is a reasonable starting point. With very small corpora (< 500 sentences) you may want to increase the training portion to 90/5/5.

In [ ]:
# split_conllu reads the source file, shuffles sentences, and writes three output files.
# It returns a dict {"train": Path, "dev": Path, "test": Path} used in the next step.
splits = split_conllu(
    input_path="wt_sanitized.conllu",
    output_dir="winter_tale/assets/en/",
    train_ratio=0.8,
    dev_ratio=0.1,
    seed=42,
)

for name, path in splits.items():
    print(f"{name:5s}: {path}")

**Options you can adjust:**
- `shuffle=False` — keeps sentences in document order (useful if you want Act 1 in train, Act 5 in test)
- `include_test=False` — omit the test split if you plan to evaluate separately

---

## Step 2: Create the model

Calling `LanguageModel()` creates the working directory structure and generates a training configuration. It does **not** start training yet.

Each component in the pipeline can be sourced from a different pre-trained model. `en_core_web_sm` provides strong general English representations for `tok2vec` and `tagger`. The bundled `UD_BASE_MODEL` provides the starting point for `morphologizer`, `trainable_lemmatizer`, and `parser` — components that `en_core_web_sm` does not have in Universal Dependencies form.

In [ ]:
model = LanguageModel(
    model_dir="winter_tale",
    lang="en",
    gpu=0,          # use GPU; set to -1 for CPU
    base_model={
        # tok2vec and tagger: en_core_web_sm is trained on a large web corpus
        # and gives strong general English representations
        "tok2vec":              "en_core_web_sm",
        "tagger":               "en_core_web_sm",
        # morphologizer, trainable_lemmatizer, parser: sourced from a UD-trained model
        # (en_core_web_sm does not have these components in Universal Dependencies form)
        "morphologizer":        UD_BASE_MODEL,
        "trainable_lemmatizer": UD_BASE_MODEL,
        "parser":               UD_BASE_MODEL,
    },
    force=True,
)

# After this runs, winter_tale/ contains:
#   config.cfg        (the training configuration)
#   assets/en/        (where your data files live)
#   corpus/en/        (converted data will go here)
#   training/en/      (trained model will appear here)
#   metrics/en/       (evaluation output will go here)

### Changing the starting point for any component

To use a different source for any component, edit the corresponding line in the `base_model` dict in the code cell above and re-run it. For example, to use the larger `en_core_web_lg` for richer tok2vec representations:

```python
"tok2vec": "en_core_web_lg",
"tagger":  "en_core_web_lg",
```

Or to point any of the three UD components at your own trained model:

```python
"morphologizer": "path/to/your_model/model-best",
```

> **Note:** If `tok2vec` is in the dict, all other components must also be in the dict. See `README.md` for the full explanation.

---

## Step 3: Copy and convert your data

First copy the data files into the model's `assets/` folder (this creates an archival copy), then convert them to spaCy's binary format for fast loading during training.

In [6]:
# The ** unpacks the dict from split_conllu() — equivalent to passing
# train=..., dev=..., test=... as keyword arguments.
model.copy_assets(**splits)

✔ Assets copied to ..\winter_tale\assets\en


In [7]:
# Convert CONLL-U files to spaCy's binary format.
# n_sents=10 groups every 10 sentences into one training document
# (more context for the model; reduce if you run out of memory).
model.convert_assets(n_sents=10)

✔ Assets converted and saved to ..\winter_tale\corpus\en


After `convert_assets()` runs you will see `.spacy` files in `winter_tale/corpus/en/`. These are the files the training loop actually reads.

---

## Step 4: Train

Start training. A progress table prints to the console as training runs — each row appears every 200 steps and shows the current accuracy on the dev set for each component.

**What to watch for:**
- `LOSS *` values should decrease as training progresses
- `TAG_ACC`, `POS_ACC`, `MORPH_ACC`, `LEMMA_ACC`, `DEP_UAS` should increase
- Training stops automatically when dev accuracy stops improving (early stopping)

The best checkpoint (highest dev score) is saved to `training/en/model-best`. The final checkpoint is `training/en/model-last`.

In [8]:
model.train()

ℹ Pipeline: ['tok2vec', 'tagger', 'morphologizer',
'trainable_lemmatizer', 'parser']
ℹ Initial learn rate: 0.001
E    #       LOSS TOK2VEC  LOSS TAGGER  LOSS MORPH...  LOSS TRAIN...  LOSS PARSER  TAG_ACC  POS_ACC  MORPH_ACC  LEMMA_ACC  DEP_UAS  DEP_LAS  SENTS_F  SCORE 
---  ------  ------------  -----------  -------------  -------------  -----------  -------  -------  ---------  ---------  -------  -------  -------  ------
  0       0       1174.91        24.20         102.91         111.74       168.20    85.81    11.10       2.44      68.07     8.53     7.06     4.25    0.31
  6     200      63661.58      2380.15        8065.40         896.18     11202.13    91.66    88.96      80.76      72.28    79.79    70.56    94.44    0.82
 14     400      19058.81       371.65        1208.83           2.14      4459.61    90.77    88.96      83.52      72.28    81.60    72.61    94.44    0.83
 21     600      10648.92       129.05         268.68           1.04      2884.53    92.20    89.73   

> **Advanced:** Training behaviour is controlled by `config.cfg` in your model directory. To change the learning rate, maximum training steps, or patience, edit the file directly and then call `model.train()` again. The key settings are in `[training.optimizer]` and `[training]`.

---

## Step 5: Evaluate

Now evaluate the trained model against the held-out test set. This gives you an honest measure of accuracy on data the model never saw during training.

In [9]:
# Evaluates model-best against the test split discovered automatically.
model.evaluate()

ℹ Using GPU: 0

================================== Results ==================================

TOK      100.00
TAG      90.75 
POS      90.73 
MORPH    88.36 
LEMMA    65.43 
UAS      80.65 
LAS      69.32 
SENT P   92.31 
SENT R   92.31 
SENT F   92.31 
SPEED    13    


============================== MORPH (per feat) ==============================

                P        R        F
Number      87.50    96.55    91.80
Person      90.48    96.61    93.44
Poss        94.12   100.00    96.97
PronType    97.53   100.00    98.75
Degree      71.43    62.50    66.67
VerbForm    89.61    83.13    86.25
Case        97.14   100.00    98.55
Definite   100.00   100.00   100.00
Mood        73.68    50.00    59.57
Tense       95.65    62.86    75.86
Voice      100.00   100.00   100.00
Gender      83.33   100.00    90.91
NumType      0.00     0.00     0.00


=============================== LAS (per type) ===============================

                    P        R        F
nmod:poss       94.44

**Reading the results:**

| Metric | What it measures | Good range |
| --- | --- | --- |
| TAG | Penn Treebank POS accuracy | 90–95% |
| POS | Universal POS accuracy | 90–95% |
| MORPH | Morphological feature accuracy | 80–95% |
| LEMMA | Lemmatization accuracy | 85–95% |
| UAS | Dependency structure (ignoring labels) | 75–90% |
| LAS | Dependency structure + label | 65–85% |

For a small corpus like one Shakespeare play, expect scores in the lower end of these ranges. More training data is the most reliable way to improve.

Full results (including per-feature and per-relation breakdowns) are saved to `metrics/en/en.json`.

---

## Step 6: Use your model

You can load and use the trained model directly without packaging it first.

In [ ]:
import spacy

nlp = spacy.load("winter_tale/training/en/model-best")

doc = nlp("If you shall chance, Camillo, to visit Bohemia on the like occasion whereon my services are now on foot.")

for token in doc:
    print(f"{token.text:20s}  POS: {token.pos_:8s}  TAG: {token.tag_:6s}  LEMMA: {token.lemma_}")

To use your model with Lexos's tokenizer, pass the path to `make_doc()`:

```python
from lexos import tokenizer
doc = tokenizer.make_doc(text, model="../winter_tale/training/en/model-best")
```

---

## Optional: Package the model

Packaging creates a pip-installable distribution that can be shared or installed in another environment. This step is optional for personal use but useful for distribution.

In [ ]:
model.package(
    input_dir="winter_tale/training/en/model-best",
    output_dir="winter_tale/packages",
    name="shakespeare_sm",
    version="1.0.0",
    force=True,
)

# After packaging:
# Load with:    spacy.load("winter_tale/packages/en_shakespeare_sm-1.0.0")
# Install with: pip install winter_tale/packages/en_shakespeare_sm-1.0.0/dist/en_shakespeare_sm-1.0.0.tar.gz

---

## Troubleshooting

**Config errors before training:** Call `debug_config()` with the path to your config file:

```python
from language_model import debug_config
debug_config("../winter_tale/config.cfg")
```

**Data errors before training:** Call `debug_data()`. Note that this will call `sys.exit(1)` if errors are found — run it in a separate cell or script:

```python
from language_model import debug_data
debug_data("../winter_tale/config.cfg")
```

**LEMMA scores dropped after fine-tuning:** The trainable lemmatizer needs more examples than other components to learn reliably. With fewer than ~500 training sentences, it often performs worse than the stock model's rule-based lemmatizer. Adding more training data is the fix.

**GPU not working:** Verify with:
```python
from thinc.api import prefer_gpu
print(prefer_gpu())  # should print True
```
If it prints `False`, re-check the GPU setup steps in `README.md`.